# EDA — LLM Classification Finetuning
Exploring Chatbot Arena preference data: label balance, position bias, verbosity bias, per-model win rates, and turn counts.

Run from the repo root or `notebooks/` (paths resolve via `src/data.py`).

In [ ]:
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent / 'src') if pathlib.Path.cwd().name == 'notebooks' else str(pathlib.Path.cwd() / 'src'))
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from data import load_train, target_labels, TARGETS
sns.set_theme(style='whitegrid')
df = load_train()
df['label'] = target_labels(df)
df['n_turns'] = df['prompt_list'].map(len)
df['len_a'] = df['response_a_text'].str.len()
df['len_b'] = df['response_b_text'].str.len()
print(df.shape)
df[['model_a','model_b'] + TARGETS].head()

## 1. Label balance & position bias

In [ ]:
dist = df['label'].value_counts(normalize=True).rename({0:'model_a',1:'model_b',2:'tie'}).sort_index()
print(dist)
print(f"\nPosition bias (a wins - b wins): {dist['model_a'] - dist['model_b']:+.4f}")
dist.plot.bar(title='Outcome distribution', ylabel='share'); plt.show()

## 2. Verbosity bias — does the longer response win?

In [ ]:
non_tie = df[df['label'] != 2].copy()
non_tie['longer_won'] = np.where(
    non_tie['label'] == 0, non_tie['len_a'] > non_tie['len_b'], non_tie['len_b'] > non_tie['len_a'])
print(f"Among non-ties, the LONGER response won {non_tie['longer_won'].mean():.1%} of the time")
non_tie['len_ratio'] = np.log1p(non_tie['len_a']) - np.log1p(non_tie['len_b'])
sns.histplot(data=non_tie, x='len_ratio', hue='label', bins=60, element='step')
plt.title('log(len_a) - log(len_b) by winner (0=a, 1=b)'); plt.show()

## 3. Per-model win rates

In [ ]:
# Stack both sides into long form: each battle contributes two model-outcome rows.
a = df[['model_a','winner_model_a','winner_tie']].rename(columns={'model_a':'model','winner_model_a':'win'})
b = df[['model_b','winner_model_b','winner_tie']].rename(columns={'model_b':'model','winner_model_b':'win'})
long = pd.concat([a.assign(tie=a['winner_tie']), b.assign(tie=b['winner_tie'])])
agg = long.groupby('model').agg(battles=('win','size'), win_rate=('win','mean'), tie_rate=('tie','mean'))
agg = agg[agg['battles'] >= 200].sort_values('win_rate', ascending=False)
print(agg.head(20))
agg['win_rate'].head(20).plot.barh(figsize=(6,6), title='Win rate (models with >=200 battles)'); plt.gca().invert_yaxis(); plt.show()

## 4. Turn counts & tie behavior

In [ ]:
print('Turns per battle:'); print(df['n_turns'].describe())
print('\nTie rate by turn count:')
print(df.groupby(df['n_turns'].clip(upper=6))['label'].apply(lambda s: (s==2).mean()))